In [1]:
import kwant
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eig


In [8]:
#参数
dela = 0.3
t = 1
af =0
a = 1
U = 0.4
mz = 1
mu = 0.5
hc=0#100
h =hc * np.sqrt(mu**2 + dela**2)
#0.5*np.sqrt(h**2 - dela**2)
#L = 11
chaodaojiao = np.pi / 2
saimanjiao = 0#np.pi / 4

#矩阵信息
sx = np.array([[0, 1], [1, 0]], complex)
sy = np.array([[0, -1j], [1j, 0]], complex)
sz = np.array([[1, 0], [0, -1]], complex)
s0 = np.eye(2, dtype=complex)

#左边矩阵信息
HL_block=-(mu-2*t)*s0 + h*np.cos(0)*sx + h*np.sin(0)*sy
Delta_L=dela * np.exp(1j*chaodaojiao/2) * 1j * sy
H_L_onsite=np.block([
        [ HL_block,        Delta_L        ],
        [ Delta_L.conj().T, -HL_block.conj() ]
    ])
H_L_right_to_left_hop_block=-1*(t*s0+1j*af*sz/a)
H_L_right_to_left_hop=np.block([
        [ H_L_right_to_left_hop_block,        np.zeros((2,2))],
        [ np.zeros((2,2)), -H_L_right_to_left_hop_block.conj() ]
    ])

#中间矩阵信息
H_center_to_L=np.block([
        [ -t*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , t*s0 ]
    ])
H_center_block=(U-mu+2*t)*s0+mz*sz
H_center=np.block([
        [ H_center_block,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , -H_center_block.conj()  ]
    ])
    
H_center_right_to_left_hop=np.block([
        [ -t*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , t*s0 ]
    ])
H_R_to_center=np.block([
        [ -t*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , t*s0 ]
    ])

#右边矩阵信息
HR_block=-(mu-2*t)*s0 + h*np.cos(saimanjiao)*sx + h*np.sin(saimanjiao)*sy
Delta_R=dela * np.exp(-1j*chaodaojiao/2) * 1j * sy
H_R_onsite=np.block([
        [ HR_block,        Delta_R        ],
        [ Delta_R.conj().T, -HR_block.conj() ]
    ])
H_R_right_to_left_hop_block=-1*(t*s0+1j*af/a*sz)
H_R_right_to_left_hop=np.block([
        [ H_R_right_to_left_hop_block,        np.zeros((2,2))],
        [ np.zeros((2,2)), -H_R_right_to_left_hop_block.conj() ]
    ])
H_q=H_center
T_12= H_center_right_to_left_hop.conj().T
H_l= H_L_onsite
T_l= H_L_right_to_left_hop
H_r= H_R_onsite
T_r=H_R_right_to_left_hop.conj().T
T_LD=H_center_to_L
T_RD= H_R_to_center
N=10
q=2
#E=0


In [ ]:
def gr_L(T_l, A_l, check_tol=1e-6):


    N = T_l.shape[0]
    I = np.eye(N)

    Tmat = np.block([
        [np.linalg.inv(T_l) @ A_l, -np.linalg.inv(T_l) @ T_l.conj().T],
        [I, np.zeros((N, N))]
    ])

    eigvals, eigvecs = eig(Tmat)


    idx = np.argsort(np.abs(eigvals))
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    lambdas = eigvals[:N]
    vecs = eigvecs[:, :N]

    S1 = vecs[:N, :] 
    S2 = vecs[N:, :]  


    gL = np.linalg.inv(A_l - T_l @ S1 @ np.linalg.inv(S2))
    check = (A_l - T_l @ gL @ T_l.conj().T) @ gL - I
    max_err = np.max(np.abs(check))
    if max_err > check_tol:
        raise RuntimeError(
            f"Self-consistency violated: max |Δ| = {max_err:e}"
        )
    return gL
#A_l = (50 + 1j*1e-6) * np.eye(H_l.shape[0]) - H_l  
#gr_L(T_l, A_l)
def zinengr_L(T_LD_wei ,gr_L_wei):
    return T_LD_wei.conj().T @ gr_L_wei @ T_LD_wei

def Gr_DD(H_q,H_l,H_r,  T_12,T_LD,T_l,T_RD,T_r,  N,E,eta=1e-6):
    d = H_q.shape[0]
    I = np.eye(d, dtype=complex)

    A_l = (E + 1j*eta) * np.eye(H_l.shape[0]) - H_l               
    Sigma_L = zinengr_L(T_LD, gr_L(A_l, T_l) )     


    A_r = (E + 1j*eta) * np.eye(H_r.shape[0]) - H_r
    Sigma_R = zinengr_L(T_RD, gr_L(A_r, T_r))

    dim = N * d
    A_DD = np.zeros((dim, dim), dtype=complex)

    for i in range(N):
        if i == 0:
            Aii = E*I - H_q - Sigma_L
        elif i == N-1:
            Aii = E*I - H_q - Sigma_R
        else:
            Aii = E*I - H_q

        A_DD[i*d:(i+1)*d, i*d:(i+1)*d] = Aii

        if i < N-1:
            A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d] = -T_12

        if i > 0:
            A_DD[i*d:(i+1)*d, (i-1)*d:i*d] = -T_12.conj().T

    G_DD_r = np.linalg.inv(A_DD)
    return G_DD_r , A_DD ,Sigma_R ,Sigma_L 

def Gr_DD(H_q, H_l, H_r, T_12, T_LD, T_l, T_RD, T_r, N, E, eta=3e-6):
    d = H_q.shape[0]
    I = np.eye(d, dtype=complex)
    Z = np.zeros((d, d), dtype=complex)

    EI_gai = np.block([
        [(E + 1j*eta)*s0, np.zeros((2,2))],
        [np.zeros((2,2)), (E + 1j*eta)*s0]
    ])

    A_l = EI_gai - H_l
    gcl = gr_L(T_l, A_l)
    Sigma_L = zinengr_L(T_LD, gcl)

    A_r = EI_gai - H_r
    gcr = gr_L(T_r, A_r)
    Sigma_R = zinengr_L(T_RD, gcr)


    A_DD = [[Z.copy() for _ in range(N)] for _ in range(N)]

    for i in range(N):

        if i == 0:
            A_DD[i][i] = EI_gai - H_q - Sigma_L
        elif i == N-1:
            A_DD[i][i] = EI_gai - H_q - Sigma_R
        else:
            A_DD[i][i] = EI_gai - H_q

        if i < N-1:
            A_DD[i][i+1] = -T_12

        if i > 0:
            A_DD[i][i-1] = -T_12.conj().T

    g_ii = [np.zeros((d, d), dtype=complex) for _ in range(N)]

    d_R = [np.zeros((d, d), dtype=complex) for _ in range(N)]
    c_R = [np.zeros((d, d), dtype=complex) for _ in range(N)]#第一个是0
    for i in range(N-1,-1,-1):
        if i==N-1:
            d_R[i]=A_DD[i][i]
        else:
            c_R[i+1]=-1*A_DD[i][i+1] @ np.linalg.inv(d_R[i+1])
            d_R[i]=A_DD[i][i]+c_R[i+1] @ A_DD[i+1][i]

    d_L = [np.zeros((d, d), dtype=complex) for _ in range(N)]
    c_L = [np.zeros((d, d), dtype=complex) for _ in range(N)]#最后一个是0

    for i in range(N):
        if i==0:
            d_L[i]=A_DD[i][i]
        else:
            c_L[i-1]=-1*A_DD[i][i-1] @ np.linalg.inv(d_L[i-1])
            d_L[i]=A_DD[i][i]+ c_L[i-1] @ A_DD[i-1][i]

        g_ii[i]=np.linalg.inv(-A_DD[i][i]+d_L[i]+d_R[i])

    G_DD_r = [[Z.copy() for _ in range(N)] for _ in range(N)]
    for i in range(N):
        for j in range(N):

            if i == j:
                G_DD_r[i][j] = g_ii[i]

            elif i > j:
                prod = np.eye(d, dtype=complex)
                for k in range(i-1, j-1, -1):
                    prod = prod @ c_L[k]
                G_DD_r[i][j] = g_ii[i] @ prod


            else:  
                prod = np.eye(d, dtype=complex)
                for k in range(i+1, j+1):
                    prod = prod @ c_R[k]
                G_DD_r[i][j] = g_ii[i] @ prod

    G_DD_r = np.block(G_DD_r)
    A_DD = np.block(A_DD)
    return G_DD_r , A_DD , Sigma_R , Sigma_L 
    
def G_DD_less_then(A_DD ,G_DD_r,zinengr_L_wei,zinengr_R_wei,E):  

    feimi1 = 1.0 / (8.617333262e-5 * 1e-6)
    feimi2 = 1.0 / (8.617333262e-5 * 1e-6)  
    zineng_DD_less_then = np.zeros((H_q.shape[0]*N, H_q.shape[0]*N), dtype=complex)
    zineng_DD_less_then[0:4, 0:4]=-(zinengr_L_wei - zinengr_L_wei.conj().T)*(1/(1+np.exp((E - mu) * feimi1)))
    zineng_DD_less_then[(N-1)*4:4*N, (N-1)*4:4*N]=-(zinengr_R_wei - zinengr_R_wei.conj().T)*(1/(1+np.exp((E - mu) * feimi2)))

    G_DD_less_then_wei= G_DD_r @ zineng_DD_less_then @ G_DD_r.conj().T 
    return G_DD_less_then_wei

def bufeng(G_DD_r):
    yigeshu=np.trace(G_DD_r)
    return -np.imag(yigeshu)/np.pi

def J_integral(G_DD_less_then_wei,T_12,q):
    d= T_12.shape[0]
    J=T_12 @ G_DD_less_then_wei[q*d:(q+1)*d, (q+1)*d:(q+2)*d]- T_12.conj() @ G_DD_less_then_wei[(q+1)*d:(q+2)*d, (q)*d:(q+1)*d]
    return np.real(np.trace(J))/(2*np.pi)
    
def current_energy_integral(E_set, q):
    Es = E_set
    dE = Es[1] - Es[0]

    I_total = 0.0

    for E in Es:
        G_DD_r, A_DD, Sigma_R, Sigma_L = Gr_DD(
            H_q, H_l, H_r,
            T_12, T_LD, T_l, T_RD, T_r,
            N, E
        )

        G_less = G_DD_less_then(A_DD, G_DD_r, Sigma_L, Sigma_R,E)

        I_E = J_integral(G_less, T_12, q)
        I_total += I_E

    return I_total * dE



In [ ]:

def Gr_DD_3(H_q,H_l,H_r,  T_12,T_LD,T_l,T_RD,T_r,  N,E,eta=3*1e-6):
    d = H_q.shape[0]
    I = np.eye(d, dtype=complex)

    EI_gai=np.block([
        [ (E + 1j*eta)*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , 1*(E + 1j*eta)*s0 ]
    ])

    A_l = EI_gai - H_l 
    gcl=gr_L(T_l, A_l)              
    Sigma_L = zinengr_L(T_LD, gcl )     


    A_r = EI_gai- H_r
    gcr=gr_L(T_r, A_r)
    Sigma_R = zinengr_L(T_RD, gcr)


    dim = N * d
    A_DD = np.zeros((dim, dim), dtype=complex)

    for i in range(N):
        if i == 0:
            Aii = EI_gai - H_q - Sigma_L
        elif i == N-1:
            Aii = EI_gai - H_q - Sigma_R
        else:
            Aii = EI_gai - H_q

        A_DD[i*d:(i+1)*d, i*d:(i+1)*d] = Aii

        if i < N-1:
            A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d] = -T_12

        if i > 0:
            A_DD[i*d:(i+1)*d, (i-1)*d:i*d] = -T_12.conj().T


    G_DD_r = np.zeros((dim, dim), dtype=complex)
    for i in range(N):
        if i==0:
            G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d]=np.linalg.inv(A_DD[i*d:(i+1)*d, i*d:(i+1)*d])
        else:
            G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d]=np.linalg.inv(A_DD[i*d:(i+1)*d, i*d:(i+1)*d]-A_DD[i*d:(i+1)*d, (i-1)*d:(i)*d] @ G_DD_r[(i-1)*d:(i)*d, (i-1)*d:(i)*d] @ A_DD[(i-1)*d:(i)*d, (i)*d:(i+1)*d])
    for i in range(N-2, -1, -1):
        #不明
        G_DD_r[(i+1)*d:(i+2)*d, (i)*d:(i+1)*d]= G_DD_r[(i+1)*d:(i+2)*d, (i+1)*d:(i+2)*d] @ A_DD[(i+1)*d:(i+2)*d, (i)*d:(i+1)*d] @ G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d]
        G_DD_r[(i)*d:(i+1)*d, (i+1)*d:(i+2)*d]= G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d]         @ A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d]   @ G_DD_r[(i+1)*d:(i+2)*d, (i+1)*d:(i+2)*d]

        G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d]=G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d] + G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d] @ A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d] @ G_DD_r[(i+1)*d:(i+2)*d, (i+1)*d:(i+2)*d] @ A_DD[(i+1)*d:(i+2)*d, (i)*d:(i+1)*d] @ G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d]

    return G_DD_r , A_DD , Sigma_R , Sigma_L 

def Gr_DD_1(H_q, H_l, H_r, T_12, T_LD, T_l, T_RD, T_r, N, E, eta=100*1e-6):
    d = H_q.shape[0]
    I = np.eye(d, dtype=complex)

    EI_gai=np.block([
        [ (E + 1j*eta)*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , 1*(E + 1j*eta)*s0 ]
    ])

    A_l = EI_gai - H_l 
    gcl=gr_L(T_l, A_l)              
    Sigma_L = zinengr_L(T_LD, gcl )     


    A_r = EI_gai- H_r
    gcr=gr_L(T_r, A_r)
    Sigma_R = zinengr_L(T_RD, gcr)


    dim = N * d
    A_DD = np.zeros((dim, dim), dtype=complex)

    for i in range(N):
        if i == 0:
            Aii = EI_gai - H_q - Sigma_L
        elif i == N-1:
            Aii = EI_gai - H_q - Sigma_R
        else:
            Aii = EI_gai - H_q

        A_DD[i*d:(i+1)*d, i*d:(i+1)*d] = Aii

        if i < N-1:
            A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d] = -T_12

        if i > 0:
            A_DD[i*d:(i+1)*d, (i-1)*d:i*d] = -T_12.conj().T


    G_DD_r = np.zeros((dim, dim), dtype=complex)
    for i in range(N):
        if i == 0:
            G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d] = np.linalg.inv(A_DD[i*d:(i+1)*d, i*d:(i+1)*d])
        else:
            T_left = A_DD[i*d:(i+1)*d, (i-1)*d:i*d] 
            G_prev = G_DD_r[(i-1)*d:i*d, (i-1)*d:i*d]
            T_right = A_DD[(i-1)*d:i*d, i*d:(i+1)*d]  
            correction = T_left @ G_prev @ T_right  
            G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d] = np.linalg.inv(A_DD[i*d:(i+1)*d, i*d:(i+1)*d] - correction)  

    for i in range(N-2, -1, -1):
        G_ii = G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d]
        G_next = G_DD_r[(i+1)*d:(i+2)*d, (i+1)*d:(i+2)*d]
        T_to_next = A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d]  
        T_from_next = A_DD[(i+1)*d:(i+2)*d, i*d:(i+1)*d] 
        correction = G_ii @ T_to_next @ G_next @ T_from_next @ G_ii  
        G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d] += correction  

        G_DD_r[(i+1)*d:(i+2)*d, i*d:(i+1)*d] = G_next @ T_from_next @ G_ii
        G_DD_r[i*d:(i+1)*d, (i+1)*d:(i+2)*d] = G_ii @ T_to_next @ G_next

    return G_DD_r, A_DD, Sigma_R, Sigma_L

def Gr_DD_2(H_q, H_l, H_r, T_12, T_LD, T_l, T_RD, T_r, N, E, eta=3*1e-6):
    d = H_q.shape[0]
    I = np.eye(d, dtype=complex)

    EI_gai = np.block([
        [(E + 1j*eta)*s0, np.zeros((2,2))],
        [np.zeros((2,2)), (E + 1j*eta)*s0]
    ])

    A_l = EI_gai - H_l
    gcl = gr_L(T_l, A_l)
    Sigma_L = zinengr_L(T_LD, gcl)

    A_r = EI_gai - H_r
    gcr = gr_L(T_r, A_r)
    Sigma_R = zinengr_L(T_RD, gcr)

    # Build lists for block tridiagonal components
    a_diag = [None] * N
    a_to_right = [None] * (N-1)  # A[i, i+1]
    a_to_left = [None] * (N-1)   # A[i+1, i]

    for i in range(N):
        if i == 0:
            a_diag[i] = EI_gai - H_q - Sigma_L
        elif i == N-1:
            a_diag[i] = EI_gai - H_q - Sigma_R
        else:
            a_diag[i] = EI_gai - H_q

        if i < N-1:
            a_to_right[i] = -T_12
            a_to_left[i] = -T_12.conj().T

    # Forward sweep (left elimination)
    d_L = [None] * N
    c_L = [None] * (N-1)
    d_L[0] = a_diag[0].copy()
    for i in range(1, N):
        c_L[i-1] = -a_to_left[i-1] @ np.linalg.inv(d_L[i-1])
        d_L[i] = a_diag[i] + c_L[i-1] @ a_to_right[i-1]

    # Backward sweep (right elimination)
    d_R = [None] * N
    c_R = [None] * (N-1)
    d_R[N-1] = a_diag[N-1].copy()
    for i in range(N-2, -1, -1):
        c_R[i] = -a_to_right[i] @ np.linalg.inv(d_R[i+1])
        d_R[i] = a_diag[i] + c_R[i] @ a_to_left[i]

    # Compute diagonal blocks of G
    dim = N * d
    G_DD_r = np.zeros((dim, dim), dtype=complex)
    for i in range(N):
        if i == 0:
            Gii = np.linalg.inv(d_R[i])
        elif i == N-1:
            Gii = np.linalg.inv(d_L[i])
        else:
            Gii = np.linalg.inv(d_L[i] + d_R[i] - a_diag[i])
        G_DD_r[i*d:(i+1)*d, i*d:(i+1)*d] = Gii

    # Build A_DD for return if needed
    A_DD = np.zeros((dim, dim), dtype=complex)
    for i in range(N):
        A_DD[i*d:(i+1)*d, i*d:(i+1)*d] = a_diag[i]
        if i < N-1:
            A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d] = a_to_right[i]
            A_DD[(i+1)*d:(i+2)*d, i*d:(i+1)*d] = a_to_left[i]

    return G_DD_r, A_DD, Sigma_R, Sigma_L

def Gr_DD(H_q, H_l, H_r, T_12, T_LD, T_l, T_RD, T_r, N, E, eta=3e-6):
    d = H_q.shape[0]
    I = np.eye(d, dtype=complex)
    Z = np.zeros((d, d), dtype=complex)

    EI_gai = np.block([
        [(E + 1j*eta)*s0, np.zeros((2,2))],
        [np.zeros((2,2)), (E + 1j*eta)*s0]
    ])

    A_l = EI_gai - H_l
    gcl = gr_L(T_l, A_l)
    Sigma_L = zinengr_L(T_LD, gcl)

    A_r = EI_gai - H_r
    gcr = gr_L(T_r, A_r)
    Sigma_R = zinengr_L(T_RD, gcr)


    A_DD = [[Z.copy() for _ in range(N)] for _ in range(N)]

    for i in range(N):

        if i == 0:
            A_DD[i][i] = EI_gai - H_q - Sigma_L
        elif i == N-1:
            A_DD[i][i] = EI_gai - H_q - Sigma_R
        else:
            A_DD[i][i] = EI_gai - H_q

        if i < N-1:
            A_DD[i][i+1] = -T_12

        if i > 0:
            A_DD[i][i-1] = -T_12.conj().T

    g_ii = [np.zeros((d, d), dtype=complex) for _ in range(N)]

    d_R = [np.zeros((d, d), dtype=complex) for _ in range(N)]
    c_R = [np.zeros((d, d), dtype=complex) for _ in range(N)]#第一个是0
    for i in range(N-1,-1,-1):
        if i==N-1:
            d_R[i]=A_DD[i][i]
        else:
            c_R[i+1]=-1*A_DD[i][i+1] @ np.linalg.inv(d_R[i+1])
            d_R[i]=A_DD[i][i]+c_R[i+1] @ A_DD[i+1][i]

    d_L = [np.zeros((d, d), dtype=complex) for _ in range(N)]
    c_L = [np.zeros((d, d), dtype=complex) for _ in range(N)]#最后一个是0

    for i in range(N):
        if i==0:
            d_L[i]=A_DD[i][i]
        else:
            c_L[i-1]=-1*A_DD[i][i-1] @ np.linalg.inv(d_L[i-1])
            d_L[i]=A_DD[i][i]+ c_L[i-1] @ A_DD[i-1][i]

        g_ii[i]=np.linalg.inv(-A_DD[i][i]+d_L[i]+d_R[i])

    G_DD_r = [[Z.copy() for _ in range(N)] for _ in range(N)]
    for i in range(N):
        for j in range(N):

            if i == j:
                G_DD_r[i][j] = g_ii[i]

            elif i > j:
                prod = np.eye(d, dtype=complex)
                for k in range(i-1, j-1, -1):
                    prod = prod @ c_L[k]
                G_DD_r[i][j] = g_ii[i] @ prod


            else:  
                prod = np.eye(d, dtype=complex)
                for k in range(i+1, j+1):
                    prod = prod @ c_R[k]
                G_DD_r[i][j] = g_ii[i] @ prod

    G_DD_r = np.block(G_DD_r)
    A_DD = np.block(A_DD)
    return G_DD_r , A_DD , Sigma_R , Sigma_L 

In [ ]:
#T_l不可逆
def gr_L(T_l, A_l, tol=1e-8):
    N = T_l.shape[0]
    I = np.eye(N)

    A = np.block([
        [np.zeros((N, N)), I],
        [-T_l.conj().T, A_l]
    ])

    B = np.block([
        [I, np.zeros((N, N))],
        [np.zeros((N, N)), T_l]
    ])

    eigvals, eigvecs = eig(A, B)

    lambdas = []
    modes = []

    for i, lam in enumerate(eigvals):
        if np.abs(lam) < 1 - tol:   # 衰减模式
            x = eigvecs[:N, i]
            x /= np.linalg.norm(x)
            lambdas.append(lam)
            modes.append(x)

    X = np.column_stack(modes)
    Lambda = np.diag(lambdas)
    #print(Lambda.shape,X.shape)
    F = X @ Lambda @ np.linalg.inv(X)

    gL = np.linalg.inv(A_l - T_l @ F)

    return gL

In [ ]:
def gr_L(T_l, A_l, check_tol=1e-6):


    N = T_l.shape[0]
    I = np.eye(N)

    Tmat = np.block([
        [np.linalg.inv(T_l) @ A_l, -np.linalg.inv(T_l) @ T_l.conj().T],
        [I, np.zeros((N, N))]
    ])

    eigvals, eigvecs = eig(Tmat)


    idx = np.argsort(np.abs(eigvals))
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    lambdas = eigvals[:N]
    vecs = eigvecs[:, :N]

    S1 = vecs[:N, :] 
    S2 = vecs[N:, :]  


    gL = np.linalg.inv(A_l - T_l @ S1 @ np.linalg.inv(S2))
    check = (A_l - T_l @ gL @ T_l.conj().T) @ gL - I
    max_err = np.max(np.abs(check))
    if max_err > check_tol:
        raise RuntimeError(
            f"Self-consistency violated: max |Δ| = {max_err:e}"
        )
    return gL

In [14]:
def gr_L(T_l, A_l, tol=1e-8, check_tol=1e-6):
    
    N = T_l.shape[0]
    I = np.eye(N, dtype=complex)

    A = np.block([
        [np.zeros((N, N)), I],
        [-T_l.conj().T, A_l]
    ])

    B = np.block([
        [I, np.zeros((N, N))],
        [np.zeros((N, N)), T_l]
    ])

    eigvals, eigvecs = eig(A, B)


    idx = np.argsort(np.abs(eigvals))
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    lambdas = []
    modes = []

    for i in range(N):
        lam = eigvals[i]
        x = eigvecs[:N, i]

        norm = np.linalg.norm(x)
        if norm < tol:
            raise RuntimeError(f"Eigenvector {i} has near-zero norm")

        x /= norm
        lambdas.append(lam)
        modes.append(x)

    if len(lambdas) != N:
        raise RuntimeError(
            f"Number of decaying modes = {len(lambdas)} ≠ N = {N}"
        )

    X = np.column_stack(modes)
    Lambda = np.diag(lambdas)

    try:
        F = X @ Lambda @ np.linalg.inv(X)
    except np.linalg.LinAlgError:
        raise RuntimeError("X is singular when constructing F")

    cond_T = np.linalg.cond(T_l.conj().T)
    if cond_T > 1e12:
        raise RuntimeError(f"T_l^† is numerically singular, cond={cond_T:e}")

    gL = F @ np.linalg.inv(T_l.conj().T)

    check = (A_l - T_l @ gL @ T_l.conj().T) @ gL - I
    max_err = np.max(np.abs(check))
#    print(max_err)

    if max_err > check_tol:
        raise RuntimeError(
            f"Self-consistency violated: max |Δ| = {max_err:e}"
        )
    return gL
A_l = (0.1 + 1j*1e-6) * np.eye(H_l.shape[0]) - H_l  
gr_L(T_l, A_l)

array([[0.00362766-3.66811598e-08j, 0.00271072+2.46567003e-03j,
        0.00360481-5.55216802e-04j, 0.00252154-2.63756131e-03j],
       [0.00271068-2.46572904e-03j, 0.00362766-3.66811188e-08j,
        0.00263751-2.52159117e-03j, 0.00055514-3.60482375e-03j],
       [0.00360482+5.55144803e-04j, 0.00263756+2.52153967e-03j,
        0.00361519-3.66813852e-08j, 0.00314389-1.82549898e-03j],
       [0.00252159+2.63750981e-03j, 0.00055522+3.60481270e-03j,
        0.00314393+1.82543997e-03j, 0.00361519-3.66814262e-08j]])

In [ ]:
import numpy as np

def build_center_chain_A(N, H_q, T_12, E, eta=1e-6, Sigma_L=None, Sigma_R=None):
    """
    构造 (E + iη)I - H_big 的 BdG tight-binding 矩阵
    """

    dim = H_q.shape[0]   # 4
    H_big = np.zeros((dim*N, dim*N), dtype=complex)

    for i in range(N):
        # onsite
        H_big[i*dim:(i+1)*dim, i*dim:(i+1)*dim] = H_q.copy()

        # hopping
        if i < N - 1:
            H_big[i*dim:(i+1)*dim, (i+1)*dim:(i+2)*dim] = T_12
            H_big[(i+1)*dim:(i+2)*dim, i*dim:(i+1)*dim] = T_12.conj().T

    # self-energy
    if Sigma_L is not None:
        H_big[0:dim, 0:dim] += Sigma_L

    if Sigma_R is not None:
        H_big[-dim:, -dim:] += Sigma_R

    # Dyson matrix
    A = (E + 1j*eta) * np.eye(dim*N, dtype=complex) - H_big

    return A


In [ ]:
def gr_L(T_l, A_l, check_tol=1e-6):


    N = T_l.shape[0]
    I = np.eye(N)

    Tmat = np.block([
        [np.linalg.inv(T_l) @ A_l, -np.linalg.inv(T_l) @ T_l.conj().T],
        [I, np.zeros((N, N))]
    ])

    eigvals, eigvecs = eig(Tmat)


    idx = np.argsort(np.abs(eigvals))
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    lambdas = eigvals[:N]
    vecs = eigvecs[:, :N]

    S1 = vecs[:N, :] 
    S2 = vecs[N:, :]  


    gL = np.linalg.inv(A_l - T_l @ S1 @ np.linalg.inv(S2))
    check = (A_l - T_l @ gL @ T_l.conj().T) @ gL - I
    max_err = np.max(np.abs(check))
 #   print(max_err)
    if max_err > check_tol:
        raise RuntimeError(
            f"Self-consistency violated: max |Δ| = {max_err:e},{E}"
        )
    return gL
A_l = (-10 + 1j*1e-6) * np.eye(H_l.shape[0]) - H_l  
gr_L(T_l, A_l)
def zinengr_L(T_LD_wei ,gr_L_wei):
    return T_LD_wei.conj().T @ gr_L_wei @ T_LD_wei


def Gr_DD(H_q,H_l,H_r,  T_12,T_LD,T_l,T_RD,T_r,  N,E,eta=1e-6):
    d = H_q.shape[0]
    I = np.eye(d, dtype=complex)

    A_l = (E + 1j*eta) * np.eye(H_l.shape[0]) - H_l 
    gcl=gr_L(T_l, A_l)              
    Sigma_L = zinengr_L(T_LD, gcl )     


    A_r = (E + 1j*eta) * np.eye(H_r.shape[0]) - H_r
    gcr=gr_L(T_l, A_l)
    Sigma_R = zinengr_L(T_RD, gcr)

    dim = N * d
    A_DD = np.zeros((dim, dim), dtype=complex)

    for i in range(N):
        if i == 0:
            Aii = (E + 1j*eta)*I - H_q - Sigma_L
        elif i == N-1:
            Aii = (E + 1j*eta)*I - H_q - Sigma_R
        else:
            Aii = (E + 1j*eta)*I - H_q

        A_DD[i*d:(i+1)*d, i*d:(i+1)*d] = Aii

        if i < N-1:
            A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d] = -T_12

        if i > 0:
            A_DD[i*d:(i+1)*d, (i-1)*d:i*d] = -T_12.conj().T

    G_DD_r = np.linalg.inv(A_DD)
    return G_DD_r , A_DD ,Sigma_R ,Sigma_L 

def bufeng(G_DD_r):
    yigeshu=np.trace(G_DD_r)
    return -np.imag(yigeshu)/np.pi

def find_peaks_simple(E_set, threshold=0.1, relative=False):
    A_E = []
    for E in E_set:
        G_DD_r, _, _, _ = Gr_DD(
            H_q, H_l, H_r,
            T_12, T_LD, T_l, T_RD, T_r,
            N, E
        )
        A_E.append(bufeng(G_DD_r))
    E_set = np.asarray(E_set)
    A_E = np.asarray(A_E)

    if not np.all(np.isfinite(A_E)):
        raise RuntimeError("A(E) contains NaN or Inf")
    if relative:
        threshold = threshold * np.max(A_E)
    peak_indices = []
    for i in range(1, len(A_E) - 1):
        if (A_E[i] - A_E[i-1] > threshold) and (A_E[i] - A_E[i+1] > threshold):
            peak_indices.append(i)
    return E_set[peak_indices]


def G_DD_less_then(A_DD ,G_DD_r,zinengr_L_wei,zinengr_R_wei,E):  

    feimi1 = 1.0 /(3.5*dela*0.5)
    feimi2 = 1.0 /(3.5*dela*0.5)
    zineng_DD_less_then = np.zeros((H_q.shape[0]*N, H_q.shape[0]*N), dtype=complex)
    zineng_DD_less_then[0:4, 0:4]=-(zinengr_L_wei - zinengr_L_wei.conj().T)*(1/(1+np.exp((E - mu) * feimi1)))
    zineng_DD_less_then[(N-1)*4:4*N, (N-1)*4:4*N]=-(zinengr_R_wei - zinengr_R_wei.conj().T)*(1/(1+np.exp((E - mu) * feimi2)))

    G_DD_less_then_wei= G_DD_r @ zineng_DD_less_then @ G_DD_r.conj().T 
    return G_DD_less_then_wei


def J_integral(G_DD_less_then_wei,T_12,q):
    d= T_12.shape[0]
    J=T_12 @ G_DD_less_then_wei[q*d:(q+1)*d, (q+1)*d:(q+2)*d]- T_12.conj() @ G_DD_less_then_wei[(q+1)*d:(q+2)*d, (q)*d:(q+1)*d]
    return np.real(np.trace(J))/(2*np.pi)
    
def current_energy_integral(E_set, q):
    Es = E_set
    dE = Es[1] - Es[0]

    I_total = 0.0

    for E in Es:
        G_DD_r, A_DD, Sigma_R, Sigma_L = Gr_DD(
            H_q, H_l, H_r,
            T_12, T_LD, T_l, T_RD, T_r,
            N, E
        )

        G_less = G_DD_less_then(A_DD, G_DD_r, Sigma_L, Sigma_R,E)

        I_E = J_integral(G_less, T_12, q)
        I_total += I_E

    return I_total * dE

E_set = np.linspace(-dela*1, dela*1, 1000)
A_E = []

for E in E_set:
    G_DD_r, _, _, _ = Gr_DD(
        H_q, H_l, H_r,
        T_12, T_LD, T_l, T_RD, T_r,
        N, E
    )
    A_E.append(bufeng(G_DD_r))

plt.plot(E_set, A_E)
plt.xlabel("E")
plt.ylabel(r"$-\mathrm{Im}\,\mathrm{Tr}\,G^r/\pi$")
plt.show()

A_E = np.array(A_E)
E_set = np.array(E_set)

threshold = 0.1
peak_indices = []

for i in range(1, len(A_E) - 1):
    if (A_E[i] - A_E[i-1] > threshold) and (A_E[i] - A_E[i+1] > threshold):
        peak_indices.append(i)

peak_energies = E_set[peak_indices]
peak_values = A_E[peak_indices]

print("峰位置 E =", peak_energies)
phi_set = np.linspace(0, 2*np.pi, 10)     # 超导相位
E_set = np.linspace(-dela, dela, 1000)   # 能量窗口
phi_list = []
E_peak_list = []
for phi in phi_set:

    # ===== 更新超导相位 =====
    chaodaojiao = phi

    # 左 lead
    Delta_L = dela * np.exp(1j*chaodaojiao/2) * 1j * sy
    H_L_onsite = np.block([
        [ HL_block,          Delta_L ],
        [ Delta_L.conj().T, -HL_block.conj() ]
    ])

    # 右 lead
    Delta_R = dela * np.exp(-1j*chaodaojiao/2) * 1j * sy
    H_R_onsite = np.block([
        [ HR_block,          Delta_R ],
        [ Delta_R.conj().T, -HR_block.conj() ]
    ])

    # 更新 lead Hamiltonian
    H_l = H_L_onsite
    H_r = H_R_onsite

    # ===== 找峰 =====
    peak_energies = find_peaks_simple(
        E_set,
        threshold=0.1,
        relative=True
    )

    # ===== 存数据 =====
    for E_peak in peak_energies:
        phi_list.append(phi)
        E_peak_list.append(E_peak)
plt.figure(figsize=(6,4))
plt.scatter(phi_list, E_peak_list, s=6)
plt.xlabel(r"Superconducting phase $\phi$")
plt.ylabel(r"Peak energy $E$")
plt.xlim(0, 2*np.pi)
plt.ylim(-dela, dela)
plt.show()



In [1]:
def Gr_DD(H_q, H_l, H_r,
          T_12, T_LD, T_l, T_RD, T_r,
          N, E, eta=1e-6):

    d = H_q.shape[0]
    I = np.eye(d, dtype=complex)

 
    A_l = (E + 1j*eta) * np.eye(H_l.shape[0]) - H_l
    gcl = gr_L(T_l, A_l)
    Sigma_L = zinengr_L(T_l.conj().T, gcl)

    A_r = (E + 1j*eta) * np.eye(H_r.shape[0]) - H_r
    gcr = gr_L(T_r, A_r)
    Sigma_R = zinengr_L(T_r.conj().T, gcr)


    NL = 2          # 左 lead 显式位点
    NR = 2          # 右 lead 显式位点
    N_tot = NL + N + NR

    dim = N_tot * d
    A_DD = np.zeros((dim, dim), dtype=complex)

    # ===== 构造 A_DD =====
    for i in range(N_tot):

        # ---------- onsite ----------
        if i==0:
            Aii = (E + 1j*eta)*I - H_l- Sigma_L
        elif 0 < i < NL:

            Aii = (E + 1j*eta)*I - H_l

        elif  NL <=i < NL + N:
            Aii = (E + 1j*eta)*I - H_q

        elif NL + N <= i < N_tot-1:

            Aii = (E + 1j*eta)*I - H_r

        elif i== N_tot-1:
            Aii = (E + 1j*eta)*I - H_r- Sigma_R

        A_DD[i*d:(i+1)*d, i*d:(i+1)*d] = Aii

        # ---------- hopping ----------
        if i < N_tot - 1:

            if i < NL - 1:
                T = T_l                     # 左 lead 内

            elif i == NL - 1:
                T = T_LD.conj().T           # 左 lead → 中心

            elif i < NL + N - 1:
                T = T_12                    # 中心内部

            elif i == NL + N - 1:
                T = T_RD                    # 中心 → 右 lead

            else:
                T = T_r                     # 右 lead 内

            A_DD[i*d:(i+1)*d, (i+1)*d:(i+2)*d] = -T
            A_DD[(i+1)*d:(i+2)*d, i*d:(i+1)*d] = -T.conj().T

    G_DD_r = np.linalg.inv(A_DD)
    return G_DD_r, A_DD, Sigma_R, Sigma_L